# 🧠🤖 每日学习 - 第4周-Day3：高级RAG技术

## 📚 主题：混合检索、重排序、查询改写

今天我们来探索RAG系统的三大核心技术：
- 🔍 **混合检索**：结合向量和关键词搜索
- ⚡ **重排序**：提升检索精度
- 🔄 **查询改写**：优化查询策略

这些技术能把RAG系统的召回率从70%提升到90%+！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Droid Sans Fallback', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("🎯 环境准备完成，开始探索高级RAG技术！")

## 🔍 混合检索：向量的力量 + 关键词的精准

混合检索结合了两种搜索方式的优点：
- **向量检索**：语义相似度，理解"like"的概念
- **BM25关键词检索**：精确匹配，找到 exact match

使用RRF（Reciprocal Rank Fusion）算法融合结果

In [ ]:
# 混合检索示例
def reciprocal_rank_fusion(vector_scores, keyword_scores, k=60):
    """RRF算法：融合向量检索和关键词检索结果"""
    fused_scores = {}
    
    # 向量分数转换
    for doc_id, score in vector_scores.items():
        rank = list(vector_scores.keys()).index(doc_id) + 1
        fused_scores[doc_id] = score / (k + rank)
    
    # 关键词分数融合
    for doc_id, score in keyword_scores.items():
        rank = list(keyword_scores.keys()).index(doc_id) + 1
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + score / (k + rank)
    
    return dict(sorted(fused_scores.items(), key=lambda x: x[1], reverse=True))

# 示例数据
documents = [
    "美华糖水店的草莓大福新鲜出炉",
    "今日特供：芒果布丁和红豆沙",
    "招牌红豆沙汤圆，用料十足",
    "季节限定：草莓系列甜品"
]

query = "草莓味的甜品"

print("📄 文档集合：")
for i, doc in enumerate(documents):
    print(f"{i+1}. {doc}")
print(f"\n🔍 查询：{query}")

# 模拟向量检索分数（语义相似度）
vector_scores = {
    0: 0.9,  # 草莓大福 - 高语义相关
    1: 0.3,  # 芒果布丁 - 低语义相关
    2: 0.2,  # 红豆沙汤圆 - 低语义相关
    3: 0.8   # 草莓系列 - 高语义相关
}

# 模拟BM25关键词检索分数
keyword_scores = {
    0: 0.8,  # 包含"草莓"
    1: 0.1,  # 不包含"草莓"
    2: 0.1,  # 不包含"草莓"
    3: 0.6   # 包含"草莓"
}

# 使用RRF融合
fused_results = reciprocal_rank_fusion(vector_scores, keyword_scores)

print("\n🎯 混合检索结果（RRF融合）：")
for i, (doc_id, score) in enumerate(fused_results.items()):
    print(f"{i+1}. 文档{doc_id}: {score:.3f} - {documents[doc_id]}")

## ⚡ 重排序：让初筛结果更精准

重排序使用Cross-Encoder模型对文档-查询对进行精确评估，比单纯的向量相似度更准确。

Cross-Encoder的优势：
- 考虑文档和查询的交互信息
- 能捕捉细粒度的语义匹配
- 精度比向量检索高20-30%

In [ ]:
# 模拟Cross-Encoder重排序
def cross_encoder_simulate(query, documents):
    """模拟Cross-Encoder重排序分数"""
    # 这里用规则模拟实际模型的效果
    scores = []
    
    for doc in documents:
        score = 0
        
        # 关键词匹配加分
        if "草莓" in doc and "草莓" in query:
            score += 0.4
        
        # 语义匹配加分
        if "甜品" in doc and "甜品" in query:
            score += 0.3
            
        # 特定产品匹配
        if "大福" in doc and "甜品" in query:
            score += 0.2
            
        # 长度匹配（避免过短文档）
        if len(doc) > 10:
            score += 0.1
            
        scores.append(score)
    
    return scores

# 计算重排序分数
rerank_scores = cross_encoder_simulate(query, documents)

# 对比不同方法的排序结果
original_order = list(range(len(documents)))
vector_order = sorted(range(len(documents)), key=lambda i: vector_scores[i], reverse=True)
keyword_order = sorted(range(len(documents)), key=lambda i: keyword_scores[i], reverse=True)
rerank_order = sorted(range(len(documents)), key=lambda i: rerank_scores[i], reverse=True)
hybrid_order = list(fused_results.keys())

print("📊 不同检索方法效果对比：")
print("\n📄 原始文档顺序：" + " → ".join([f"{i+1}.{documents[i]}" for i in original_order]))
print("🔵 向量检索顺序：" + " → ".join([f"{i+1}.{documents[i]}" for i in vector_order]))
print("🔴 关键词检索顺序：" + " → ".join([f"{i+1}.{documents[i]}" for i in keyword_order]))
print("⚡ 重排序顺序：" + " → ".join([f"{i+1}.{documents[i]}" for i in rerank_order]))
print("🎯 混合检索顺序：" + " → ".join([f"{i+1}.{documents[i]}" for i in hybrid_order]))

# 可视化对比
plt.figure(figsize=(12, 8))

# 方法名称
methods = ['原始', '向量检索', '关键词检索', '重排序', '混合检索']
colors = ['gray', 'blue', 'red', 'orange', 'green']

# 为每种方法创建分数条
for i, (method, color) in enumerate(zip(methods, colors)):
    scores = []
    if method == '原始':
        scores = [0.5] * len(documents)  # 平均分
    elif method == '向量检索':
        scores = [vector_scores.get(j, 0) for j in range(len(documents))]
    elif method == '关键词检索':
        scores = [keyword_scores.get(j, 0) for j in range(len(documents))]
    elif method == '重排序':
        scores = rerank_scores
    elif method == '混合检索':
        scores = [fused_results.get(j, 0) for j in range(len(documents))]
    
    x_pos = np.arange(len(documents)) + i * 0.15
    plt.bar(x_pos, scores, width=0.1, color=color, alpha=0.7, label=method)

plt.xlabel('文档编号')
plt.ylabel('相关度分数')
plt.title('不同检索方法效果对比')
plt.xticks(np.arange(len(documents)) + 0.4, [f'文档{i+1}' for i in range(len(documents))])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔄 查询改写：让模糊问题变清晰

查询改写主要有两种方法：

### 1. HyDE (Hypothetical Document Embeddings)
- 先假设答案，生成理想文档
- 再用这个文档去检索

### 2. 多查询扩展 (Multi-Query)
- 把一个查询扩展成多个相关查询
- 每个角度检索，最后合并结果

In [ ]:
# 模拟HyDE查询改写
def hyde_query_rewrite(query):
    """HyDE：假设文档生成"""
    # 根据查询生成假设的理想文档
    if "草莓" in query and "甜品" in query:
        hypothetical_doc = "美味的草莓甜品，新鲜草莓制作，口感香甜，是下午茶的绝佳选择"
    elif "芒果" in query:
        hypothetical_doc = "香甜芒果布丁，芒果果肉丰富，口感顺滑，热带水果的代表"
    else:
        hypothetical_doc = "精致甜品，用料新鲜，口感香甜，让人回味无穷"
    
    return hypothetical_doc

# 模拟多查询扩展
def multi_query_expansion(query):
    """多查询扩展"""
    queries = [query]  # 原始查询
    
    # 同义词扩展
    if "草莓" in query:
        queries.extend([query.replace("草莓", "鲜草莓"), 
                        query.replace("甜品", "甜点")])
    
    # 场景扩展
    if "今天" in query:
        queries.append(query.replace("今天", "今日"))
        queries.append(query.replace("今天", "当前供应"))
    
    return list(set(queries))  # 去重

# 演示查询改写效果
original_query = "今天有没有草莓味的甜品"

print(f"🔍 原始查询：{original_query}")
print()

# HyDE改写
hyde_doc = hyde_query_rewrite(original_query)
print(f"📝 HyDE改写：生成假设文档")
print(f"""   "{hyde_doc}""")
print()

# 多查询扩展
expanded_queries = multi_query_expansion(original_query)
print(f"🔄 多查询扩展：生成 {len(expanded_queries)} 个相关查询")
for i, eq in enumerate(expanded_queries):
    print(f"   {i+1}. {eq}")
print()

# 可视化查询改写效果
plt.figure(figsize=(15, 5))

# 原始查询vs改写查询的覆盖范围
original_keywords = set(["今天", "有", "没有", "草莓", "味", "甜品"])
hyde_keywords = set(["美味", "草莓", "甜品", "新鲜", "制作", "香甜", "下午茶", "选择"])
multi_keywords = set()
for eq in expanded_queries:
    multi_keywords.update(eq.split())

# 创建维恩图数据
from matplotlib_venn import venn3

plt.subplot(1, 3, 1)
venn3([original_keywords, hyde_keywords, multi_keywords], 
      set_labels=['原始查询', 'HyDE文档', '多查询扩展'])
plt.title('关键词覆盖范围对比')

# 查询改写效果模拟
plt.subplot(1, 3, 2)
query_types = ['原始查询', 'HyDE改写', '多查询']
recall_rates = [0.65, 0.82, 0.78]  # 模拟召回率提升
precision_rates = [0.72, 0.85, 0.80]  # 模拟精确率提升

x = np.arange(len(query_types))
width = 0.35

plt.bar(x - width/2, recall_rates, width, label='召回率', alpha=0.8)
plt.bar(x + width/2, precision_rates, width, label='精确率', alpha=0.8)
plt.xlabel('查询类型')
plt.ylabel('比率')
plt.title('查询改写效果')
plt.xticks(x, query_types)
plt.legend()
plt.ylim(0, 1)

# 综合提升效果
plt.subplot(1, 3, 3)
improvement_areas = ['关键词匹配', '语义理解', '覆盖范围', '歧义消除']
original_scores = [0.6, 0.5, 0.7, 0.4]
hyde_scores = [0.7, 0.9, 0.8, 0.7]
multi_scores = [0.8, 0.7, 0.9, 0.6]

for i, area in enumerate(improvement_areas):
    plt.scatter([0, 1, 2], [original_scores[i], hyde_scores[i], multi_scores[i]], 
                s=100, alpha=0.7)
    plt.text(0, original_scores[i]+0.02, area, ha='center', fontsize=8)
    plt.text(1, hyde_scores[i]+0.02, area, ha='center', fontsize=8)
    plt.text(2, multi_scores[i]+0.02, area, ha='center', fontsize=8)

plt.xticks([0, 1, 2], ['原始', 'HyDE', '多查询'])
plt.ylabel('效果分数')
plt.title('不同场景下的改进效果')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 RAG系统性能对比分析

让我们通过一个完整的例子来对比不同RAG策略的效果：

In [ ]:
# 完整RAG系统效果模拟
def simulate_rag_system(query, documents, strategy="basic"):
    """模拟不同RAG策略的效果"""
    
    # 基础：简单向量检索
    if strategy == "basic":
        scores = [cosine_similarity([[query]], [[doc]])[0][0] for doc in documents]
        
    # 混合检索
    elif strategy == "hybrid":
        vector_scores = [cosine_similarity([[query]], [[doc]])[0][0] for doc in documents]
        keyword_scores = [1 if "草莓" in doc else 0.2 for doc in documents]
        
        # RRF融合
        scores = []
        for i in range(len(documents)):
            v_score = vector_scores[i] / (60 + i + 1)
            k_score = keyword_scores[i] / (60 + i + 1)
            scores.append(v_score + k_score)
            
    # 完整高级RAG
    elif strategy == "advanced":
        # 1. 查询改写
        rewritten_queries = multi_query_expansion(query)
        
        all_scores = []
        
        # 2. 多查询检索
        for rq in rewritten_queries:
            vector_scores = [cosine_similarity([[rq]], [[doc]])[0][0] for doc in documents]
            keyword_scores = [1 if "草莓" in doc else 0.2 for doc in documents]
            
            # RRF融合
            fused_scores = []
            for i in range(len(documents)):
                v_score = vector_scores[i] / (60 + i + 1)
                k_score = keyword_scores[i] / (60 + i + 1)
                fused_scores.append(v_score + k_score)
            
            all_scores.append(fused_scores)
        
        # 3. 结果合并
        scores = []
        for i in range(len(documents)):
            avg_score = np.mean([scores[i] for scores in all_scores])
            scores.append(avg_score)
            
        # 4. 重排序
        rerank_scores = cross_encoder_simulate(query, documents)
        # 加权融合
        final_scores = []
        for i in range(len(documents)):
            final_score = 0.7 * scores[i] + 0.3 * rerank_scores[i]
            final_scores.append(final_score)
        scores = final_scores
    # 返回排序后的文档索引和分数
    sorted_indices = np.argsort(scores)[::-1]
    return sorted_indices, scores

# 测试不同策略
strategies = ["基础RAG", "混合检索", "高级RAG"]
results = {}

for strategy in strategies:
    strategy_key = strategy.replace("RAG", "").replace("检索", "").lower()
    indices, scores = simulate_rag_system(query, documents, strategy_key)
    results[strategy] = (indices, scores)
    
    print(f"\n🎯 {strategy} 结果：")
    for i, doc_idx in enumerate(indices):
        print(f"   {i+1}. {documents[doc_idx]} (分数: {scores[doc_idx]:.3f})")

# 可视化性能对比
plt.figure(figsize=(15, 10))

# 1. 不同策略的召回率对比
plt.subplot(2, 3, 1)
relevant_docs = [0, 3]  # 相关文档索引
strategy_names = list(results.keys())
recall_scores = []

for strategy in strategies:
    indices, scores = results[strategy]
    retrieved_top_k = set(indices[:2])  # 取前2个
    relevant = set(relevant_docs)
    recall = len(retrieved_top_k.intersection(relevant)) / len(relevant)
    recall_scores.append(recall)

plt.bar(strategy_names, recall_scores, color=['lightblue', 'lightgreen', 'lightcoral'])
plt.title('召回率对比 (Top-2)')
plt.ylabel('召回率')
plt.ylim(0, 1)
for i, score in enumerate(recall_scores):
    plt.text(i, score + 0.02, f'{score:.1%}', ha='center', va='bottom')

# 2. 不同策略的精确率对比
plt.subplot(2, 3, 2)
precision_scores = []

for strategy in strategies:
    indices, scores = results[strategy]
    retrieved_top_k = set(indices[:2])
    relevant = set(relevant_docs)
    precision = len(retrieved_top_k.intersection(relevant)) / len(retrieved_top_k)
    precision_scores.append(precision)

plt.bar(strategy_names, precision_scores, color=['lightblue', 'lightgreen', 'lightcoral'])
plt.title('精确率对比 (Top-2)')
plt.ylabel('精确率')
plt.ylim(0, 1)
for i, score in enumerate(precision_scores):
    plt.text(i, score + 0.02, f'{score:.1%}', ha='center', va='bottom')

# 3. 分数分布对比
plt.subplot(2, 3, 3)
for i, (strategy, (indices, scores)) in enumerate(results.items()):
    plt.scatter(range(len(scores)), scores, label=strategy, alpha=0.7, s=50)
plt.xlabel('文档排名')
plt.ylabel('相关度分数')
plt.title('不同策略的分数分布')
plt.legend()
plt.grid(True, alpha=0.3)

# 4. 策略复杂度对比
plt.subplot(2, 3, 4)
complexity_scores = [1, 3, 8]  # 模拟计算复杂度
performance_scores = [recall_scores[0], recall_scores[1], recall_scores[2]]  # 性能提升

plt.scatter(complexity_scores, performance_scores, s=100, alpha=0.7)
for i, strategy in enumerate(strategy_names):
    plt.annotate(strategy, (complexity_scores[i], performance_scores[i]), 
                 xytext=(5, 5), textcoords='offset points', fontsize=9)
plt.xlabel('计算复杂度')
plt.ylabel('召回率提升')
plt.title('复杂度vs性能权衡')
plt.grid(True, alpha=0.3)

# 5. 实际应用场景分析
plt.subplot(2, 3, 5)
scenarios = ['简单查询', '模糊查询', '专业术语', '多意图']
improvement_over_basic = [
    0.05,  # 简单查询提升不大
    0.25,  # 模糊查询提升明显
    0.30,  # 专业术语提升很多
    0.20   # 多意图中等提升
]

plt.bar(scenarios, improvement_over_basic, color=['gold', 'orange', 'red', 'purple'])
plt.title('高级RAG在不同场景的提升幅度')
plt.ylabel('召回率提升')
plt.xticks(rotation=45)
for i, imp in enumerate(improvement_over_basic):
    plt.text(i, imp + 0.01, f'+{imp:.1%}', ha='center', va='bottom')

# 6. 最终建议
plt.subplot(2, 3, 6)
plt.text(0.5, 0.8, '🎯 RAG策略选择建议', ha='center', va='center', fontsize=14, weight='bold')
plt.text(0.1, 0.6, '• 基础RAG：适合简单场景，速度快', ha='left', va='center', fontsize=10)
plt.text(0.1, 0.45, '• 混合检索：平衡性能和复杂度', ha='left', va='center', fontsize=10)
plt.text(0.1, 0.3, '• 高级RAG：关键业务必备，效果最佳', ha='left', va='center', fontsize=10)
plt.text(0.1, 0.15, '💡 建议：从混合检索开始，根据效果逐步升级', ha='left', va='center', fontsize=10)
plt.axis('off')
plt.title('策略选择指南')

plt.tight_layout()
plt.show()

print("\n🎉 RAG系统优化完成！总结：")
print("✅ 混合检索提升精确匹配能力")
print("✅ 重排序优化最终结果排序")
print("✅ 查询改写解决模糊查询问题")
print("✅ 综合应用可将召回率提升30%+！")

## 💡 课后练习

### 📝 思考题

1. **场景分析**：美华糖水店的顾客查询中，哪些场景最适合混合检索？
   - 精确产品名称（如"草莓大福"）
   - 模糊描述（如"甜甜的水果"）
   - 场景需求（如"适合儿童的甜品"）

2. **技术选型**：如果糖水店要建立RAG系统推荐甜品，你会选择哪种RAG策略？
   - 考虑查询类型：产品名称vs场景需求
   - 考虑数据特点：产品描述vs用户评价
   - 考算性能要求：响应时间vs准确率

3. **改进思路**：除了今天学到的三种技术，还有哪些方法可以进一步提升RAG效果？
   - 多轮对话的上下文理解
   - 用户历史偏好的个性化检索
   - 实时库存信息结合

### 🚀 实战建议

周末代码实战时，可以尝试以下进阶功能：

1. **构建混合检索模块**：同时使用向量和BM25检索
2. **实现RRF融合算法**：融合多路检索结果
3. **添加查询改写功能**：HyDE或多查询扩展
4. **集成重排序模型**：使用预训练的Cross-Encoder

### 🎓 学习要点回顾

今天我们深入学习了RAG的三大核心技术：

🔍 **混合检索** = 向量语义 + 关键词精确
⚡ **重排序** = Cross-encoder精排提升精度
🔄 **查询改写** = HyDE/多查询解决模糊问题

这些技术的组合使用，能让你的RAG系统从"能用"升级到"好用"，在实际业务场景中发挥巨大价值！